[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/e_series/e_series.ipynb)

# The E Series

This example constructs the exceptional root systems E6, E7 and E8 from nested sets of generating mirrors in eight dimensions. Each root system is a finite configuration of equal-length mirror normals preserved by reflection across every member, and adding one generating mirror at a time extends E6 into E7 and E7 into E8 so that all three appear nested inside a single Coxeter-plane projection.

Each generating mirror is modelled as an oriented plane with reflection map `mirror >> Plane`, and composing a system's simple reflections gives its Coxeter motion as one linear map on planes. Averaging that motion with its metric adjoint `(action + action.adjoint()) / 2` isolates its invariant rotation planes in a single eigensolve, while summing the root dyads `roots * (roots | Plane)` yields the orthogonal projector onto each root span. Because E7 composes seven reflections, its Coxeter motion turns three perpendicular planes and reverses one remaining axis rather than forming an even rotor. John Stembridge's [Coxeter-plane account](https://websites.umich.edu/~jrs/coxplane.html) derives the circular orbit projections, and the [MIT root-system notes](https://math.mit.edu/~dav/roots.pdf) situate the exceptional diagrams among the classical families.

In Weyl-group matrix notation, each reflection is written as $s_i(v) = v - 2\frac{\langle \alpha_i, v\rangle}{\langle \alpha_i, \alpha_i\rangle}\alpha_i$ from the Cartan matrix $A_{ij} = 2\langle \alpha_i, \alpha_j\rangle / \langle \alpha_i, \alpha_i\rangle$, and the invariant planes of $w = s_1 \cdots s_r$ are extracted by complexifying $\mathbb{R}^8$ to diagonalize $w$ into conjugate eigenvalues $e^{\pm 2\pi i m_k / h}$ for the Coxeter exponents $m_k$ ($1, 4, 5, 7, 8, 11$ at $h = 12$; $1, 5, 7, 9, 11, 13, 17$ at $h = 18$ with $e^{\pi i} = -1$ at $m = 9$; and $1, 7, 11, 13, 17, 19, 23, 29$ at $h = 30$).

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure

from examples.math.e_series import render

np.set_printoptions(precision=6, suppress=True)

ga = Algebra("a+b+c+d+e+f+g+h+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Plane = ga.gatype.vector()                                                    # Plane+
PlaneMap = ga.gatype((Plane, Plane))                                          # Plane+ <- Plane+

## Generating mirrors

Eight simple roots specify the generating mirrors in the shared ambient space. Connected nodes in the Dynkin diagram meet at 120 degrees and unconnected nodes are perpendicular; taking the first six, seven or eight mirrors generates E6, E7 or E8.

In [ ]:
names = ("E6", "E7", "E8")
ranks = np.array([6, 7, 8])
periods = np.array([12, 18, 30])

simple_roots = stack([
    (mv.a + mv.h - mv.b - mv.c - mv.d - mv.e - mv.f - mv.g) / 2,
    mv.a + mv.b,
    mv.b - mv.a,
    mv.c - mv.b,
    mv.d - mv.c,
    mv.e - mv.d,
    mv.f - mv.e,
    mv.g - mv.f,
])                                                                            # [8] Plane+

display(Image(filename=str(save_figure(
    render.diagram_figure(simple_roots, ranks, names), "e_series_diagram",
))));

Teal marks the six mirrors that generate E6, amber adds the seventh mirror for E7, and purple adds the eighth mirror for E8.

## Root orbits

Composing one reflection per simple root forms a Coxeter motion of period 12, 18 or 30. Reflecting each simple root in the mirrors that precede it provides one seed per generator whose disjoint orbits exhaust the 72, 126 and 240 roots.

In [ ]:
def orbit_starts(simple: Plane) -> Iterator[Plane]:
    """Yields one starting root (`[] Plane+`) per generator, reflected in the mirrors preceding it."""
    preceding = Plane                                                         # [] Plane+ <- Plane+
    for root in simple:
        yield preceding(root)                                                 # [] Plane+
        preceding = preceding(root.normalized() >> Plane)                     # [] Plane+ <- Plane+


def orbit(action: PlaneMap, seeds: Plane, n_steps: int) -> Iterator[Plane]:
    """Yields the successive images (`[...] Plane+`) of `seeds` under `n_steps` applications of `action`."""
    for _ in range(n_steps):
        yield seeds                                                           # [...] Plane+
        seeds = action(seeds)                                                 # [...] Plane+

In [ ]:
def root_system(simple: Plane, n_steps: int) -> tuple[PlaneMap, Plane]:
    """The generating reflections of `simple` compose into the Coxeter motion (`[] Plane+ <- Plane+`) and carry one seed per generator through all roots (`[n_steps, ...] Plane+`)."""
    reflections = simple.normalized() >> Plane                                # [...] Plane+ <- Plane+
    # Right composition matches the prefix order used to choose the starting roots.
    action = reduce(lambda left, right: left(right), reflections)             # [] Plane+ <- Plane+
    seeds = stack(orbit_starts(simple))                                       # [...] Plane+
    roots = stack(orbit(action, seeds, n_steps))                              # [n_steps, ...] Plane+
    return action, roots


systems = tuple(root_system(simple_roots[:n_rank], n_steps)
                for n_rank, n_steps in zip(ranks, periods))
root_systems = tuple(roots for _, roots in systems)

## Shared Coxeter-plane projection

Averaging an orthogonal motion with its adjoint cancels the skew turn in every invariant plane and leaves each plane scaled by the cosine of its turn angle. Paired eigendirections recover those planes from slowest to fastest, with the slowest turn of E8 defining a single Coxeter plane in which all three nested systems appear together.

In [ ]:
def rotation_planes(action: PlaneMap, n_rank: int) -> tuple[Plane, Plane, Scalar]:
    """Symmetric averaging of `action` recovers its orthogonal rotation planes (`[...] Plane+`, `[...] Plane+`) and turn angles (`[...] Scalar+`) from slowest to fastest."""
    symmetric = (action + action.adjoint()) / 2                               # [] Plane+ <- Plane+
    cosines, eigenvectors = symmetric.eigh()                                  # [8] Scalar+, Plane+

    # The eigenvectors outside the root span have cosine one and sort last.
    # Paired cosines describe turns; a lone minus one describes a reflection.
    plane_cosines = cosines[n_rank - 1:0:-2]                                  # [...] Scalar+
    plane_x = eigenvectors[n_rank - 1:0:-2]                                   # [...] Plane+
    plane_y = (action(plane_x) - plane_cosines * plane_x).normalized()        # [...] Plane+
    return plane_x, plane_y, plane_cosines.arccos()


plane_frames = tuple(rotation_planes(action, n_rank)
                     for (action, _), n_rank in zip(systems, ranks))
plane_x, plane_y, _ = plane_frames[-1]                                        # [4] Plane+, Plane+, Scalar+

display(Image(filename=str(save_figure(
    render.nested_roots_figure(root_systems, plane_x[0], plane_y[0], names),
    "e_series_nested",
))));

Teal dots mark the 72 roots of E6, amber dots mark the 54 roots added by E7, and purple dots mark the 114 roots added by E8, while faint strokes join nearest neighbours in eight dimensions.

## Projectors onto the root spans

Each root contributes a symmetric dyad `root * (root | Plane)` whose sum acts uniformly across every direction of the root span. Dividing by that uniform eigenvalue produces the orthogonal projector onto each system's span, and composing adjacent projectors confirms that each span sits inside the next.

In [ ]:
# Root lengths squared are two; each system has rank times period roots.
spans = stack([
    (roots * (roots | Plane)).sum(axis=(0, 1)) / (2 * n_steps)
    for roots, n_steps in zip(root_systems, periods)
])                                                                            # [3] Plane+ <- Plane+
shared_spans = spans[:-1](spans[1:])                                          # [2] Plane+ <- Plane+

print("Root-span dimensions, E6 / E7 / E8:")
print(spans.trace().to_array())
print("Shared dimensions, E6 with E7 / E7 with E8:")
print(shared_spans.trace().to_array())

## Odd reflection axis in E7

The Coxeter motions of E6 and E8 factor completely into turns across three and four perpendicular planes. Subtracting the turning-plane projector from the full root-span projector isolates the remaining seventh axis of E7, which reverses sign at every step because seven reflections have odd parity.

In [ ]:
# The plane dyads account for every turning direction.
turning_spans = stack([
    (frame_x * (frame_x | Plane) + frame_y * (frame_y | Plane)).sum(axis=0)
    for frame_x, frame_y, _ in plane_frames
])                                                                            # [3] Plane+ <- Plane+
reversing_spans = spans - turning_spans                                       # [3] Plane+ <- Plane+

print("Reversed dimensions, E6 / E7 / E8:")
print(reversing_spans.trace().to_array())
winding_counts = []
for name, (_, _, angles), n_steps in zip(names, plane_frames, periods):
    winding_rates = angles * n_steps / (2 * np.pi)                            # [...] Scalar+
    winding_counts.append(winding_rates)
    print(f"{name}: turns in each plane over one complete orbit")
    print(winding_rates.to_array())

In [ ]:
# checks
for roots, n_roots in zip(root_systems, (72, 126, 240)):
    assert np.prod(roots.shape) == n_roots
    assert ((roots.scalar_norm_squared() - 2).abs() < 1e-10).all()
residual_spans = spans(spans) - spans                                         # [3] Plane+ <- Plane+
assert (residual_spans.adjoint()(residual_spans).trace() < 1e-10).all()
nested_residuals = shared_spans - spans[:-1]                                  # [2] Plane+ <- Plane+
assert (nested_residuals.adjoint()(nested_residuals).trace() < 1e-10).all()
assert ((reversing_spans.trace() - np.array([0, 1, 0])).abs() < 1e-10).all()
for winding_rates, expected in zip(winding_counts, ([1, 4, 5], [1, 5, 7], [1, 7, 11, 13])):
    assert ((winding_rates - np.array(expected)).abs() < 1e-10).all()